# 03.01 — Ingesta de NYC Yellow Taxi Trip Records

## Objetivo

En este notebook se implementará la primera etapa de ingesta del proyecto utilizando
los registros públicos de viajes de **NYC Yellow Taxi**, publicados por la
New York City Taxi & Limousine Commission (TLC).

Los datos se encuentran disponibles en archivos **Parquet mensuales**, lo que permite
trabajar con un volumen considerable de información utilizando Apache Spark.

Durante este notebook se realizará:

1. Definición de las fuentes de datos.
2. Descarga automatizada de los archivos mensuales.
3. Organización de los datos dentro de la capa `raw`.
4. Lectura de múltiples archivos Parquet mediante PySpark.
5. Validación inicial del esquema y volumen de información.
6. Exploración básica del dataset antes de iniciar las transformaciones.

Para esta primera versión del proyecto se utilizarán los registros correspondientes
al año **2024**.

## 1. Importaciones

In [1]:
from pathlib import Path
from urllib.request import urlretrieve

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

## 2. Crear la Sesión de Spark

In [2]:
spark = (
    SparkSession.builder
    .appName("03_01_nyc_taxi_ingestion")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

In [3]:
print("=" * 60)
print("INFORMACIÓN DEL ENTORNO SPARK")
print("=" * 60)

print(f"Spark version: {spark.version}")
print(f"Application name: {spark.sparkContext.appName}")
print(f"Master: {spark.sparkContext.master}")
print(f"Default parallelism: {spark.sparkContext.defaultParallelism}")

INFORMACIÓN DEL ENTORNO SPARK
Spark version: 4.1.3
Application name: 03_01_nyc_taxi_ingestion
Master: local[*]
Default parallelism: 12


## 3. Configuración de la fuente de datos

In [4]:
YEAR = 2024

BASE_URL = (
    "https://d37ci6vzurychx.cloudfront.net/"
    "trip-data/yellow_tripdata_{year}-{month:02d}.parquet"
)

MONTHS = range(1, 13)

In [5]:
for month in MONTHS:
    url = BASE_URL.format(year=YEAR, month=month)
    print(url)

https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet
https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-02.parquet
https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-03.parquet
https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-04.parquet
https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-05.parquet
https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-06.parquet
https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-07.parquet
https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-08.parquet
https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-09.parquet
https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet
https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-11.parquet
https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-12.parquet


## 4. Capa de datos raw

Los archivos descargados se almacenarán inicialmente sin modificar dentro de
`data/raw`.

Esta capa conserva los datos en un formato lo más cercano posible a la fuente
original y funciona como punto de entrada al pipeline.

En etapas posteriores del proyecto se crearán capas transformadas y optimizadas
para análisis.

In [6]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "taxi" / f"year={YEAR}"

RAW_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(f"Directorio raw: {RAW_DATA_DIR}")

Directorio raw: c:\Users\derec\Desktop\Proyectos\BigData_project\data\raw\taxi\year=2024


## 5. Función de descarga

In [7]:
def download_taxi_file(year: int, month: int, output_dir: Path) -> Path:
    """
    Descarga un archivo mensual de NYC Yellow Taxi.

    Parameters
    ----------
    year : int
        Año correspondiente al archivo.

    month : int
        Mes correspondiente al archivo.

    output_dir : Path
        Directorio donde se almacenará el archivo.

    Returns
    -------
    Path
        Ruta local del archivo descargado.
    """

    filename = f"yellow_tripdata_{year}-{month:02d}.parquet"

    url = BASE_URL.format(
        year=year,
        month=month
    )

    output_path = output_dir / filename

    if output_path.exists():
        print(f"[SKIP] {filename} ya existe.")
        return output_path

    print(f"[DOWNLOAD] {filename}")

    urlretrieve(
        url,
        output_path
    )

    return output_path

## 6. Primera prueba: descargar solamente enero

In [8]:
test_file = download_taxi_file(
    year=YEAR,
    month=1,
    output_dir=RAW_DATA_DIR
)

print(test_file)

[DOWNLOAD] yellow_tripdata_2024-01.parquet
c:\Users\derec\Desktop\Proyectos\BigData_project\data\raw\taxi\year=2024\yellow_tripdata_2024-01.parquet


In [9]:
print(f"Archivo existe: {test_file.exists()}")

if test_file.exists():
    size_mb = test_file.stat().st_size / (1024 ** 2)
    print(f"Tamaño: {size_mb:.2f} MB")

Archivo existe: True
Tamaño: 47.65 MB


## 7. Lectura inicial con Spark

Antes de descargar el año completo, se utilizará el archivo correspondiente
a enero como muestra para validar que Spark puede leer correctamente los datos
Parquet proporcionados por NYC TLC.

En esta etapa verificaremos:

- esquema detectado;
- número de registros;
- columnas disponibles;
- primeras observaciones;
- rango temporal de los viajes.

In [10]:
taxi_january_df = (
    spark.read
    .parquet(str(test_file))
)

taxi_january_df

DataFrame[VendorID: int, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: bigint, trip_distance: double, RatecodeID: bigint, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double]

## 8. Inspeccionar el esquema

In [11]:
taxi_january_df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [12]:
print(f"Número de columnas: {len(taxi_january_df.columns)}")

Número de columnas: 19


In [13]:
taxi_january_df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee']

## 9. Número de viajes

In [14]:
january_rows = taxi_january_df.count()

print(f"Viajes registrados en enero de {YEAR}: {january_rows:,}")

Viajes registrados en enero de 2024: 2,964,624


## 10. Visualización inicial

In [15]:
taxi_january_df.show(
    n=5,
    truncate=False
)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|2       |2024-01-01 00:57:55 |2024-01-01 01:17:43  |1              |1.72         |1         |N                 |186         |79          |2           |17.7       |1.0  |0.5    |0.0      

In [16]:
taxi_january_df.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "PULocationID",
    "DOLocationID",
    "fare_amount",
    "tip_amount",
    "total_amount"
).show(
    n=10,
    truncate=False
)

+--------------------+---------------------+---------------+-------------+------------+------------+-----------+----------+------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|PULocationID|DOLocationID|fare_amount|tip_amount|total_amount|
+--------------------+---------------------+---------------+-------------+------------+------------+-----------+----------+------------+
|2024-01-01 00:57:55 |2024-01-01 01:17:43  |1              |1.72         |186         |79          |17.7       |0.0       |22.7        |
|2024-01-01 00:03:00 |2024-01-01 00:09:36  |1              |1.8          |140         |236         |10.0       |3.75      |18.75       |
|2024-01-01 00:17:06 |2024-01-01 00:35:01  |1              |4.7          |236         |79          |23.3       |3.0       |31.3        |
|2024-01-01 00:36:38 |2024-01-01 00:44:56  |1              |1.4          |79          |211         |10.0       |2.0       |17.0        |
|2024-01-01 00:46:51 |2024-01-01 00:52:57

Esta segunda vista ya empieza a mostrar claramente la estructura de nuestro problema.

Cada fila representa esencialmente:

```
Viaje
 │
 ├── cuándo inició
 ├── cuándo terminó
 ├── pasajeros
 ├── distancia
 ├── zona de origen
 ├── zona de destino
 ├── tarifa
 ├── propina
 └── importe total
 ```

## 11. Validar el rango temporal

In [17]:
taxi_january_df.select(
    F.min("tpep_pickup_datetime").alias("min_pickup"),
    F.max("tpep_pickup_datetime").alias("max_pickup"),
    F.min("tpep_dropoff_datetime").alias("min_dropoff"),
    F.max("tpep_dropoff_datetime").alias("max_dropoff")
).show(truncate=False)

+-------------------+-------------------+-------------------+-------------------+
|min_pickup         |max_pickup         |min_dropoff        |max_dropoff        |
+-------------------+-------------------+-------------------+-------------------+
|2002-12-31 22:59:39|2024-02-01 00:01:15|2002-12-31 23:05:41|2024-02-02 13:56:52|
+-------------------+-------------------+-------------------+-------------------+



## 12. Distribución básica de variables

Por ahora queremos verificar que los datos tengan sentido.

In [18]:
taxi_january_df.select(
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount"
).summary().show()

+-------+------------------+------------------+------------------+------------------+------------------+
|summary|   passenger_count|     trip_distance|       fare_amount|        tip_amount|      total_amount|
+-------+------------------+------------------+------------------+------------------+------------------+
|  count|           2824462|           2964624|           2964624|           2964624|           2964624|
|   mean|1.3392808966805005|3.6521691789583146|18.175061916791037|3.3358700158940002|26.801504770952707|
| stddev|0.8502816924800847|225.46257238220082| 18.94954770590526| 3.896550599806763|23.385577429672516|
|    min|                 0|               0.0|            -899.0|             -80.0|            -900.0|
|    25%|                 1|               1.0|               8.6|               1.0|             15.38|
|    50%|                 1|              1.68|              12.8|               2.7|              20.1|
|    75%|                 1|              3.11|        

## 13. Verificación rápida de valores nulos

In [19]:
null_counts = taxi_january_df.select(
    [
        F.sum(
            F.col(column).isNull().cast("int")
        ).alias(column)
        for column in taxi_january_df.columns
    ]
)

null_counts.show(
    vertical=True,
    truncate=False
)

-RECORD 0-----------------------
 VendorID              | 0      
 tpep_pickup_datetime  | 0      
 tpep_dropoff_datetime | 0      
 passenger_count       | 140162 
 trip_distance         | 0      
 RatecodeID            | 140162 
 store_and_fwd_flag    | 140162 
 PULocationID          | 0      
 DOLocationID          | 0      
 payment_type          | 0      
 fare_amount           | 0      
 extra                 | 0      
 mta_tax               | 0      
 tip_amount            | 0      
 tolls_amount          | 0      
 improvement_surcharge | 0      
 total_amount          | 0      
 congestion_surcharge  | 140162 
 Airport_fee           | 140162 



## 14. Ingesta del año completo

Una vez validada correctamente la lectura del primer archivo, se procede a
automatizar la descarga de los doce meses correspondientes a 2024.

La función de descarga evita volver a descargar archivos existentes, haciendo
que el proceso pueda ejecutarse repetidamente sin generar copias innecesarias.

In [20]:
downloaded_files = []

for month in MONTHS:

    file_path = download_taxi_file(
        year=YEAR,
        month=month,
        output_dir=RAW_DATA_DIR
    )

    downloaded_files.append(file_path)

[SKIP] yellow_tripdata_2024-01.parquet ya existe.
[DOWNLOAD] yellow_tripdata_2024-02.parquet
[DOWNLOAD] yellow_tripdata_2024-03.parquet
[DOWNLOAD] yellow_tripdata_2024-04.parquet
[DOWNLOAD] yellow_tripdata_2024-05.parquet
[DOWNLOAD] yellow_tripdata_2024-06.parquet
[DOWNLOAD] yellow_tripdata_2024-07.parquet
[DOWNLOAD] yellow_tripdata_2024-08.parquet
[DOWNLOAD] yellow_tripdata_2024-09.parquet
[DOWNLOAD] yellow_tripdata_2024-10.parquet
[DOWNLOAD] yellow_tripdata_2024-11.parquet
[DOWNLOAD] yellow_tripdata_2024-12.parquet


## 15. Validar los archivos descargados

In [21]:
print("=" * 60)
print("ARCHIVOS DISPONIBLES")
print("=" * 60)

for file_path in downloaded_files:

    size_mb = file_path.stat().st_size / (1024 ** 2)

    print(
        f"{file_path.name:<40} "
        f"{size_mb:>8.2f} MB"
    )

ARCHIVOS DISPONIBLES
yellow_tripdata_2024-01.parquet             47.65 MB
yellow_tripdata_2024-02.parquet             48.02 MB
yellow_tripdata_2024-03.parquet             57.30 MB
yellow_tripdata_2024-04.parquet             56.39 MB
yellow_tripdata_2024-05.parquet             59.66 MB
yellow_tripdata_2024-06.parquet             57.09 MB
yellow_tripdata_2024-07.parquet             49.88 MB
yellow_tripdata_2024-08.parquet             48.70 MB
yellow_tripdata_2024-09.parquet             58.34 MB
yellow_tripdata_2024-10.parquet             61.37 MB
yellow_tripdata_2024-11.parquet             57.85 MB
yellow_tripdata_2024-12.parquet             58.67 MB


In [22]:
total_size_mb = sum(
    file_path.stat().st_size
    for file_path in downloaded_files
) / (1024 ** 2)

print(f"\nTamaño total: {total_size_mb:,.2f} MB")


Tamaño total: 660.90 MB


## 16. Leer los 12 archivos con Spark

In [23]:
taxi_df = (
    spark.read
    .parquet(str(RAW_DATA_DIR))
)

In [24]:
print(f"Columnas: {len(taxi_df.columns)}")

taxi_df.printSchema()

Columnas: 19
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [25]:
total_rows = taxi_df.count()

print(
    f"Total de viajes {YEAR}: "
    f"{total_rows:,}"
)

Total de viajes 2024: 41,169,720


## 17. Una validación adicional

Como tenemos una columna temporal, vamos a comprobar cuántos registros está leyendo Spark por mes:


In [26]:
monthly_counts = (
    taxi_df
    .withColumn(
        "year",
        F.year("tpep_pickup_datetime")
    )
    .withColumn(
        "month",
        F.month("tpep_pickup_datetime")
    )
    .groupBy(
        "year",
        "month"
    )
    .count()
    .orderBy(
        "year",
        "month"
    )
)

monthly_counts.show(
    n=50,
    truncate=False
)

+----+-----+-------+
|year|month|count  |
+----+-----+-------+
|2002|12   |11     |
|2008|12   |10     |
|2009|1    |18     |
|2023|12   |10     |
|2024|1    |2964617|
|2024|2    |3007533|
|2024|3    |3582611|
|2024|4    |3514295|
|2024|5    |3723843|
|2024|6    |3539170|
|2024|7    |3076876|
|2024|8    |2979192|
|2024|9    |3633025|
|2024|10   |3833780|
|2024|11   |3646372|
|2024|12   |3668350|
|2025|1    |2      |
|2025|2    |1      |
|2025|3    |2      |
|2026|6    |2      |
+----+-----+-------+



## 18. Validación temporal del dataset

Aunque los archivos descargados corresponden nominalmente al año 2024,
la fecha contenida en cada registro debe validarse de forma independiente.

Se identificaron observaciones cuyo `tpep_pickup_datetime` se encuentra fuera
del rango esperado.

En esta etapa los registros anómalos no serán eliminados, ya que la capa `raw`
debe conservar los datos originales. Su tratamiento se realizará posteriormente
durante las transformaciones y controles de calidad.

In [28]:
START_DATE = "2024-01-01"
END_DATE = "2025-01-01"

valid_2024_condition = (
    (F.col("tpep_pickup_datetime") >= F.lit(START_DATE)) &
    (F.col("tpep_pickup_datetime") < F.lit(END_DATE))
)

In [29]:
valid_rows = (
    taxi_df
    .filter(valid_2024_condition)
    .count()
)

invalid_rows = (
    taxi_df
    .filter(~valid_2024_condition)
    .count()
)

print(f"Registros totales:      {total_rows:,}")
print(f"Registros válidos 2024: {valid_rows:,}")
print(f"Fuera de rango:         {invalid_rows:,}")

Registros totales:      41,169,720
Registros válidos 2024: 41,169,664
Fuera de rango:         56


In [30]:
invalid_pct = invalid_rows / total_rows * 100

print(
    f"Porcentaje fuera de rango: "
    f"{invalid_pct:.6f}%"
)

Porcentaje fuera de rango: 0.000136%


## 19. Examinar los registros anómalos

In [31]:
invalid_dates_df = (
    taxi_df
    .filter(~valid_2024_condition)
    .select(
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "trip_distance",
        "fare_amount",
        "total_amount"
    )
    .orderBy("tpep_pickup_datetime")
)

invalid_dates_df.show(
    n=100,
    truncate=False
)

+--------------------+---------------------+------------+------------+-------------+-----------+------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|DOLocationID|trip_distance|fare_amount|total_amount|
+--------------------+---------------------+------------+------------+-------------+-----------+------------+
|2002-12-31 16:46:07 |2002-12-31 17:24:07  |138         |263         |7.77         |37.3       |62.68       |
|2002-12-31 22:10:04 |2002-12-31 22:14:22  |170         |231         |7.55         |33.1       |45.72       |
|2002-12-31 22:17:10 |2002-12-31 22:42:24  |50          |162         |1.4          |10.0       |18.0        |
|2002-12-31 22:17:43 |2002-12-31 22:23:55  |237         |140         |0.67         |7.9        |11.9        |
|2002-12-31 22:19:28 |2003-01-01 18:58:09  |246         |68          |0.55         |7.2        |11.2        |
|2002-12-31 22:59:39 |2002-12-31 23:05:41  |170         |170         |0.63         |-6.5       |-10.5       |
|2002-12-3

## 20. distinguir fecha del viaje vs archivo de origen

In [32]:
taxi_with_source_df = (
    taxi_df
    .withColumn(
        "source_file",
        F.input_file_name()
    )
)

In [33]:
taxi_with_source_df.select(
    "tpep_pickup_datetime",
    "source_file"
).show(
    n=5,
    truncate=False
)

+--------------------+----------------------------------------------------------------------------------------------------------------+
|tpep_pickup_datetime|source_file                                                                                                     |
+--------------------+----------------------------------------------------------------------------------------------------------------+
|2024-05-01 00:59:15 |file:///c:/Users/derec/Desktop/Proyectos/BigData_project/data/raw/taxi/year=2024/yellow_tripdata_2024-05.parquet|
|2024-04-30 23:58:26 |file:///c:/Users/derec/Desktop/Proyectos/BigData_project/data/raw/taxi/year=2024/yellow_tripdata_2024-05.parquet|
|2024-05-01 00:57:17 |file:///c:/Users/derec/Desktop/Proyectos/BigData_project/data/raw/taxi/year=2024/yellow_tripdata_2024-05.parquet|
|2024-05-01 00:24:47 |file:///c:/Users/derec/Desktop/Proyectos/BigData_project/data/raw/taxi/year=2024/yellow_tripdata_2024-05.parquet|
|2024-05-01 00:11:20 |file:///c:/Users/derec/Des

## 21. Extraer el mes del archivo fuente




In [34]:
taxi_with_source_df = (
    taxi_with_source_df
    .withColumn(
        "source_month",
        F.regexp_extract(
            F.col("source_file"),
            r"yellow_tripdata_2024-(\d{2})\.parquet",
            1
        ).cast("int")
    )
    .withColumn(
        "pickup_month",
        F.month("tpep_pickup_datetime")
    )
    .withColumn(
        "pickup_year",
        F.year("tpep_pickup_datetime")
    )
)

In [35]:
taxi_with_source_df.select(
    "source_month",
    "pickup_year",
    "pickup_month"
).show(10)

+------------+-----------+------------+
|source_month|pickup_year|pickup_month|
+------------+-----------+------------+
|           5|       2024|           5|
|           5|       2024|           4|
|           5|       2024|           5|
|           5|       2024|           5|
|           5|       2024|           5|
|           5|       2024|           5|
|           5|       2024|           5|
|           5|       2024|           5|
|           5|       2024|           5|
|           5|       2024|           5|
+------------+-----------+------------+
only showing top 10 rows


In [36]:
file_date_mismatch_df = (
    taxi_with_source_df
    .filter(
        (F.col("pickup_year") != YEAR) |
        (F.col("pickup_month") != F.col("source_month"))
    )
)

In [37]:
file_date_mismatch_count = file_date_mismatch_df.count()

print(
    f"Registros cuya fecha no coincide "
    f"con el archivo mensual: "
    f"{file_date_mismatch_count:,}"
)

Registros cuya fecha no coincide con el archivo mensual: 420


## 22. Conteo por archivo físico

In [38]:
file_counts = (
    taxi_with_source_df
    .groupBy(
        "source_month"
    )
    .count()
    .orderBy(
        "source_month"
    )
)

file_counts.show(12)

+------------+-------+
|source_month|  count|
+------------+-------+
|           1|2964624|
|           2|3007526|
|           3|3582628|
|           4|3514289|
|           5|3723833|
|           6|3539193|
|           7|3076903|
|           8|2979183|
|           9|3633030|
|          10|3833771|
|          11|3646369|
|          12|3668371|
+------------+-------+



## 23. Resumen de la ingesta

In [39]:
summary_data = [
    ("Año", str(YEAR)),
    ("Archivos esperados", str(len(MONTHS))),
    ("Registros totales", f"{total_rows:,}"),
    ("Registros válidos por fecha", f"{valid_rows:,}"),
    ("Registros fuera de 2024", f"{invalid_rows:,}"),
    ("Columnas", str(len(taxi_df.columns))),
    ("Formato", "Parquet"),
]

summary_df = spark.createDataFrame(
    summary_data,
    ["metric", "value"]
)

summary_df.show(
    truncate=False
)

+---------------------------+----------+
|metric                     |value     |
+---------------------------+----------+
|Año                        |2024      |
|Archivos esperados         |12        |
|Registros totales          |41,169,720|
|Registros válidos por fecha|41,169,664|
|Registros fuera de 2024    |56        |
|Columnas                   |19        |
|Formato                    |Parquet   |
+---------------------------+----------+



## Conclusiones

La ingesta de NYC Yellow Taxi para 2024 se completó correctamente.

Se descargaron y cargaron los doce archivos mensuales en formato Parquet,
obteniendo aproximadamente 41.2 millones de registros y 19 variables.

La validación inicial mostró que una pequeña cantidad de observaciones contiene
fechas fuera del año esperado. Estos registros se conservarán en la capa `raw`
y serán tratados posteriormente durante las etapas de calidad y transformación
de datos.

También se distinguió entre el período indicado por el archivo físico y el
período derivado del timestamp del viaje, permitiendo implementar controles
adicionales de consistencia.

Con esto, los datos de viajes quedan disponibles para las siguientes etapas
del pipeline.

In [27]:
import sys

print(f"Python: {sys.executable}")
print(f"Working directory: {Path.cwd()}")
print(f"Spark master: {spark.sparkContext.master}")

Python: c:\Users\derec\anaconda3\envs\projects\python.exe
Working directory: c:\Users\derec\Desktop\Proyectos\BigData_project\notebooks
Spark master: local[*]
